# Merged BERTopic - Build

This notebook merges the 7 per-outlet BERTopic models into one shared topic space.

**Run this once after all per-outlet models are final. Then use `Merged_BERTopic_Analysis.ipynb` for all inspection.**

Steps:
1. Load per-outlet models from `1a_BERTopic/outputs/`
2. Merge via `BERTopic.merge_models()`
3. Inspect initial topic overview
4. Assign topics to all articles from `df_combined`
5. Save merged model + article assignments

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from bertopic import BERTopic
from IPython.display import display

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MIN_SIMILARITY = 0.7

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not find project root (.git)")

PROJECT_ROOT = find_project_root(Path.cwd())
OUTPUTS_DIR = PROJECT_ROOT / "1a_BERTopic" / "outputs"
MERGED_SAVE_DIR = OUTPUTS_DIR / "merged_model"

os.environ["NUMBA_CACHE_DIR"] = str(PROJECT_ROOT / ".numba_cache")

print(f"Project root: {PROJECT_ROOT}")
print(f"Models dir:   {OUTPUTS_DIR}")
print(f"Merged save:  {MERGED_SAVE_DIR}")

## 1. Load per-outlet models

In [ ]:
OUTLET_MODEL_KEYS = {
    "tagesschau":        "ts_model",
    "rt":                "rt_model",
    "antispiegel":       "as_model",
    "tichys":            "te_model",
    "nius":              "ns_model",
    "compact":           "compact_model",
    "deutschlandkurier": "dk_model",
}

OUTLET_LABELS = {
    "tagesschau":        "Tagesschau",
    "rt":                "RT DE",
    "antispiegel":       "Anti-Spiegel",
    "tichys":            "Tichys Einblick",
    "nius":              "Nius",
    "compact":           "Compact",
    "deutschlandkurier": "Deutschland-Kurier",
}

loaded_models = {}
for key, folder in OUTLET_MODEL_KEYS.items():
    model_path = OUTPUTS_DIR / folder
    model = BERTopic.load(model_path, embedding_model=EMBEDDING_MODEL)
    ti = model.get_topic_info()
    n_topics = len(ti[ti["Topic"] != -1])
    n_outlier = ti.loc[ti["Topic"] == -1, "Count"].sum() if -1 in ti["Topic"].values else 0
    total = ti["Count"].sum()
    print(f"{OUTLET_LABELS[key]:22s}  {n_topics:3d} topics  {n_outlier:4d}/{total} outliers ({100*n_outlier/total:.1f}%)")
    loaded_models[key] = model

print(f"\nLoaded {len(loaded_models)} models.")

## 2. Merge models

`BERTopic.merge_models()` computes cosine similarity between topic c-TF-IDF vectors across all outlets.
Topics with similarity ≥ `MIN_SIMILARITY` (0.7) are deduplicated into one canonical topic.
The result is a new model with a shared topic space - Tagesschau is listed first as the reference outlet.

In [ ]:
# Tagesschau first = reference outlet
models_to_merge = [
    loaded_models["tagesschau"],
    loaded_models["rt"],
    loaded_models["antispiegel"],
    loaded_models["tichys"],
    loaded_models["nius"],
    loaded_models["compact"],
    loaded_models["deutschlandkurier"],
]

print(f"Merging {len(models_to_merge)} models with min_similarity={MIN_SIMILARITY} ...")
merged_model = BERTopic.merge_models(models_to_merge, min_similarity=MIN_SIMILARITY)

topic_info = merged_model.get_topic_info()
n_merged = len(topic_info[topic_info["Topic"] != -1])
print(f"\nMerged topic count: {n_merged} substantive topics (+ outlier topic -1)")

## 3. Initial topic overview

Inspect the merged topics before any outlier reduction or renaming. Use this to decide:
- Is the number of topics reasonable?
- Are there obvious noise/junk topics to prune?
- Do you want to apply `reduce_outliers` on the merged assignments?

In [ ]:
topic_info_display = topic_info.copy()
topic_info_display["TopKeywords"] = topic_info_display["Name"].str.replace(r"^\d+_", "", regex=True)

# Show non-outlier topics sorted by size
topics_only = (
    topic_info_display[topic_info_display["Topic"] != -1]
    .sort_values("Count", ascending=False)
    .reset_index(drop=True)
)
topics_only.index += 1

print(f"Total merged topics: {len(topics_only)}")
display(topics_only[["Topic", "Count", "TopKeywords"]].head(40))

In [ ]:
# Full list - export to CSV for inspection
overview_path = OUTPUTS_DIR / "merged_topics_initial_overview.csv"
topics_only[["Topic", "Count", "TopKeywords"]].to_csv(overview_path, index=False)
print(f"Topic list saved to: {overview_path}")
display(topics_only[["Topic", "Count", "TopKeywords"]])

## 4. Assign topics to all articles

Load `df_combined` (all 7 outlets, cleaned), prepare documents, and transform them through the merged model.
This gives each article a `merged_topic` assignment for downstream analysis.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import re

DATA_PREP_DIR = PROJECT_ROOT / "data preprocessing"

# Load combined corpus
df_combined = pd.read_csv(DATA_PREP_DIR / "overall_df_combined.csv", parse_dates=["Date"])
print(f"Loaded df_combined: {len(df_combined):,} rows")
print(df_combined["source"].value_counts())

In [ ]:
# Minimal text prep: title + text, strip boilerplate patterns
MIN_TOKENS = 8
MIN_CHARS = 50

df_combined["doc_text"] = (
    df_combined["Title"].fillna("").astype(str).str.strip()
    + ". "
    + df_combined["Text"].fillna("").astype(str).str.strip()
).str.replace(r"^\. ", "", regex=True).str.strip()

# Apply length filter (same as pipeline)
mask = (
    (df_combined["doc_text"].str.split().str.len() >= MIN_TOKENS) &
    (df_combined["doc_text"].str.len() >= MIN_CHARS)
)
df_filtered = df_combined[mask].copy().reset_index(drop=True)
print(f"After length filter: {len(df_filtered):,} articles (dropped {len(df_combined) - len(df_filtered)})")

docs = df_filtered["doc_text"].tolist()

In [ ]:
# Transform: assign each article to the nearest merged topic
# This can take a few minutes for ~22k articles
print("Embedding + assigning topics ...")
topics, probs = merged_model.transform(docs)
df_filtered["merged_topic"] = topics

# Summary
n_outlier = sum(1 for t in topics if t == -1)
n_total = len(topics)
print(f"Assigned topics to {n_total:,} articles")
print(f"Outliers (topic -1): {n_outlier:,} ({100*n_outlier/n_total:.1f}%)")

## 5. Save merged model and article assignments

Set `SAVE = True` to persist. The model is needed by `Merged_BERTopic_Analysis.ipynb`.

In [ ]:
import shutil

SAVE = True

if SAVE:
    # Save model
    if MERGED_SAVE_DIR.exists():
        shutil.rmtree(MERGED_SAVE_DIR)
    merged_model.save(
        MERGED_SAVE_DIR,
        serialization="safetensors",
        save_ctfidf=True,
        save_embedding_model=EMBEDDING_MODEL,
    )
    print(f"Merged model saved to: {MERGED_SAVE_DIR}")

    # Save article assignments
    assignments_path = OUTPUTS_DIR / "merged_article_topics.parquet"
    save_cols = ["Date", "Title", "source", "merged_topic", "doc_text"]
    df_filtered[save_cols].to_parquet(assignments_path, index=False)
    print(f"Article assignments saved to: {assignments_path}")
else:
    print("SAVE=False — set to True to persist model + assignments")

## 6. Quick sanity check before handing off to analysis

Per-outlet article counts by topic. Use this to spot anything obviously wrong before running the full analysis notebook.

In [ ]:
# Source name → display label mapping
SOURCE_LABELS = {
    "Tagesschau":          "Tagesschau",
    "RT":                  "RT DE",
    "Antispiegel":         "Anti-Spiegel",
    "Tichys_Einblick":     "Tichys Einblick",
    "Nius":                "Nius",
    "Compact":             "Compact",
    "Deutschlandkurier":   "Deutschland-Kurier",
}

# Articles per outlet per topic (top 20 topics)
top_topics = (
    df_filtered[df_filtered["merged_topic"] != -1]
    .groupby("merged_topic")
    .size()
    .sort_values(ascending=False)
    .head(20)
    .index.tolist()
)

pivot = (
    df_filtered[df_filtered["merged_topic"].isin(top_topics)]
    .groupby(["merged_topic", "source"])
    .size()
    .unstack(fill_value=0)
)
pivot["TOTAL"] = pivot.sum(axis=1)
pivot = pivot.sort_values("TOTAL", ascending=False)

# Add topic keywords
topic_kw = dict(zip(topic_info["Topic"], topic_info["Name"].str.replace(r"^\d+_", "", regex=True)))
pivot.insert(0, "keywords", pivot.index.map(topic_kw))

display(pivot)